# SKILL.md Base vs. Derived Comparison

This notebook safely loads the sample `SKILL.md` files as **plain UTF-8 text only**. It does **not execute** shell commands, Python, or any other content found inside the dataset.

### What it does
- discovers skill identifiers from `base/` and `derived/`
- extracts front-matter fields (`name`, `description`) and full content
- records repository/file information and SHA-256 hashes
- compares base vs. derived text
- reports invalid/missing records with a clear reason
- saves analysis records to CSV and JSON
- runs built-in automated assertions for loader/extraction logic

### Setup / run
1. Place this notebook where it can access `sample_skills/`, or change `SAMPLE_ROOT` in the next cell.
2. Run all cells from top to bottom.
3. Outputs are written to `analysis_output/skill_comparisons.csv` and `.json`.

Only the Python standard library is required.

In [1]:
from pathlib import Path
import csv, json, hashlib, difflib, re

# Change this if your sample_skills folder is elsewhere.
CANDIDATES = [
    Path("sample_skills"),
    Path("tests/sample_skills"),
    Path("../tests/sample_skills"),
    Path("../../tests/sample_skills"),
]
SAMPLE_ROOT = next((p for p in CANDIDATES if p.exists()), CANDIDATES[0])
BASE_DIR = SAMPLE_ROOT / "base"
DERIVED_DIR = SAMPLE_ROOT / "derived"
OUTPUT_DIR = Path("analysis_output")

print("Sample root:", SAMPLE_ROOT.resolve())

Sample root: /Users/vaisnavii/Desktop/UM-Flint/Courses/CSC 580/CSC 580 - Final Project/csc580-project8/tests/sample_skills


## Safe loader and extraction functions

These functions only read text. They never execute anything contained in a `SKILL.md` file.

In [2]:
def load_text(path):
    path = Path(path)
    if not path.exists(): return None, "file_not_found"
    if not path.is_file(): return None, "not_a_file"
    try:
        return path.read_text(encoding="utf-8"), None
    except UnicodeDecodeError:
        return None, "invalid_utf8"
    except OSError as e:
        return None, f"read_error: {e}"

def extract_front_matter(text):
    result = {"name": None, "description": None}
    if not text.startswith("---"):
        return result, "missing_front_matter"
    parts = text.split("---", 2)
    if len(parts) < 3:
        return result, "malformed_front_matter"
    for line in parts[1].splitlines():
        if ":" in line:
            key, value = line.split(":", 1)
            key = key.strip()
            if key in result:
                result[key] = value.strip()
    if not result["name"]:
        return result, "missing_name"
    return result, None

def sha256_text(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

def compare_text(base_text, derived_text):
    diff = list(difflib.unified_diff(
        base_text.splitlines(), derived_text.splitlines(),
        fromfile="base/SKILL.md", tofile="derived/SKILL.md", lineterm=""
    ))
    added = [x[1:] for x in diff if x.startswith("+") and not x.startswith("+++")]
    removed = [x[1:] for x in diff if x.startswith("-") and not x.startswith("---")]
    return {
        "changed": base_text != derived_text,
        "added_line_count": len(added),
        "removed_line_count": len(removed),
        "added_lines": "\n".join(added),
        "removed_lines": "\n".join(removed),
        "diff": "\n".join(diff),
    }

## Detection Model

The detector applies regex-based rules to **added lines in the derived `SKILL.md` only**. This answers whether a newly introduced change matches a risk rule, rather than flagging behavior that was already present in the base file.

- **Command execution** — CMD-001 shell code block; CMD-002 command invocation
- **Network access** — NET-001 HTTP/HTTPS URL; NET-002 network client usage
- **Filesystem access** — FS-001 file-system command; FS-002 file-writing API
- **Credential access** — CRED-001 credential/secret reference; CRED-002 credential file reference
- **External code execution** — EXT-001 downloaded content piped to shell; EXT-002 external package installation
- **System modification** — SYS-001 privilege elevation; SYS-002 system modification


In [3]:
DETECTION_RULES = [
    ("CMD-001", "Command execution", r"```\s*(?:sh|bash|zsh|shell|powershell|cmd)\b", "shell code block detected"),
    ("CMD-002", "Command execution", r"(?im)^\s*(?:\$\s*)?(?:bash|sh|zsh|python|python3|node|npm|npx|uv|git|make|docker|kubectl)\b", "command invocation detected"),
    ("NET-001", "Network access", r"https?://[^\s)>\]}`]+", "HTTP/HTTPS URL detected"),
    ("NET-002", "Network access", r"\b(?:curl|wget|requests\.(?:get|post|put|delete)|urllib|fetch\s*\(|httpx\.|aiohttp\.|Invoke-WebRequest)\b", "network client usage detected"),
    ("FS-001", "Filesystem access", r"(?im)^\s*(?:\$\s*)?(?:cp|mv|rm|mkdir|rmdir|touch|chmod|chown|cat|ls|find)\b", "file-system command detected"),
    ("FS-002", "Filesystem access", r"\b(?:open\s*\([^\n]*(?:['\"](?:w|a|x|wb|ab)['\"])|write_text\s*\(|write_bytes\s*\(|shutil\.(?:copy|move)|Path\([^\n]*\)\.write_)\b", "file-writing API detected"),
    ("CRED-001", "Credential access", r"(?i)\b(?:credential|credentials|secret|secrets|api[_ -]?key|access[_ -]?token|auth[_ -]?token|password|passwd)\b", "credential or secret reference detected"),
    ("CRED-002", "Credential access", r"(?i)(?:\.env\b|\.aws/credentials\b|\.ssh/(?:id_rsa|id_ed25519)\b|\.netrc\b|credentials\.json\b|secrets?\.ya?ml\b)", "credential file reference detected"),
    ("EXT-001", "External code execution", r"(?i)\b(?:curl|wget)\b[^\n|]*\|\s*(?:sudo\s+)?(?:sh|bash|zsh|python|python3)\b", "downloaded content piped to a shell interpreter"),
    ("EXT-002", "External code execution", r"(?im)^\s*(?:\$\s*)?(?:(?:python\s+-m\s+)?pip(?:3)?\s+install|uv\s+(?:add|pip\s+install)|npm\s+(?:install|i)\b|yarn\s+add\b|pnpm\s+(?:add|install)\b|apt(?:-get)?\s+install\b|brew\s+install\b)", "external package installation detected"),
    ("SYS-001", "System modification", r"(?im)^\s*(?:\$\s*)?sudo\b", "privilege-elevation command detected"),
    ("SYS-002", "System modification", r"(?im)^\s*(?:\$\s*)?(?:systemctl|service|launchctl|sysctl|defaults\s+write|update-alternatives)\b", "system modification command detected"),
]

def detect_rules(text):
    """Return every detection rule matched by text, with concise evidence."""
    matches = []
    for rule_id, category, pattern, description in DETECTION_RULES:
        found = list(re.finditer(pattern, text, flags=re.MULTILINE))
        if found:
            evidence = []
            for m in found[:3]:
                line = text.count("\n", 0, m.start()) + 1
                snippet = text.splitlines()[line - 1].strip() if text.splitlines() else m.group(0)
                evidence.append(f"added line {line}: {snippet[:200]}")
            matches.append({"rule_id": rule_id, "category": category, "description": description,
                            "evidence": " | ".join(evidence)})
    return matches

def summarize_rule_matches(added_text):
    matches = detect_rules(added_text)
    return {
        "matched_rule_ids": ", ".join(m["rule_id"] for m in matches),
        "matched_risk_categories": ", ".join(dict.fromkeys(m["category"] for m in matches)),
        "rule_match_count": len(matches),
        "rule_matches": matches,
        "rule_evidence": " || ".join(f'{m["rule_id"]}: {m["evidence"]}' for m in matches),
    }


## Automated tests

These assertions act as a small automated test/fixture inside the notebook.

In [4]:
from tempfile import TemporaryDirectory

with TemporaryDirectory() as d:
    p = Path(d) / "SKILL.md"
    p.write_text("---\nname: demo\ndescription: Test skill\n---\n# Demo\n", encoding="utf-8")
    text, err = load_text(p)
    assert err is None
    meta, meta_err = extract_front_matter(text)
    assert meta_err is None
    assert meta["name"] == "demo"
    assert meta["description"] == "Test skill"

    missing_text, missing_err = load_text(Path(d) / "missing.md")
    assert missing_text is None and missing_err == "file_not_found"

    cmp = compare_text("old\n", "new\n")
    assert cmp["changed"] is True
    assert "new" in cmp["added_lines"] and "old" in cmp["removed_lines"]

print("All automated tests passed.")

All automated tests passed.


## Discover, validate, extract, and compare records

In [5]:
if not BASE_DIR.exists() or not DERIVED_DIR.exists():
    raise FileNotFoundError(
        f"Could not find base/derived folders under {SAMPLE_ROOT}. "
        "Update SAMPLE_ROOT in the setup cell."
    )

skill_ids = sorted(
    {p.name for p in BASE_DIR.iterdir() if p.is_dir()} |
    {p.name for p in DERIVED_DIR.iterdir() if p.is_dir()}
)

records = []
for skill_id in skill_ids:
    base_path = BASE_DIR / skill_id / "SKILL.md"
    derived_path = DERIVED_DIR / skill_id / "SKILL.md"
    base_text, base_error = load_text(base_path)
    derived_text, derived_error = load_text(derived_path)

    record = {
        "skill_id": skill_id,
        "repository_root": str(SAMPLE_ROOT),
        "base_path": str(base_path),
        "derived_path": str(derived_path),
    }

    if base_error or derived_error:
        record.update({
            "status": "invalid",
            "skip_reason": f"base={base_error or 'ok'}; derived={derived_error or 'ok'}",
        })
        records.append(record)
        continue

    base_meta, base_meta_error = extract_front_matter(base_text)
    derived_meta, derived_meta_error = extract_front_matter(derived_text)
    if base_meta_error or derived_meta_error:
        record.update({
            "status": "invalid",
            "skip_reason": f"base={base_meta_error or 'ok'}; derived={derived_meta_error or 'ok'}",
        })
        records.append(record)
        continue

    comparison = compare_text(base_text, derived_text)
    rule_summary = summarize_rule_matches(comparison["added_lines"])

    record.update({
        "status": "valid", "skip_reason": "",
        "base_name": base_meta["name"],
        "derived_name": derived_meta["name"],
        "base_description": base_meta["description"],
        "derived_description": derived_meta["description"],
        "base_content": base_text, "derived_content": derived_text,
        "base_sha256": sha256_text(base_text),
        "derived_sha256": sha256_text(derived_text),
        **comparison,
        **rule_summary,
    })
    records.append(record)

print(f"Processed {len(records)} skill pair(s).")
for r in records:
    print(f"{r['skill_id']}: {r['status']}, changed={r.get('changed', 'n/a')}")


Processed 5 skill pair(s).
api-report: valid, changed=True
bootstrap-helper: valid, changed=True
config-backup: valid, changed=True
release-notes-writer: valid, changed=True
repo-status-reporter: valid, changed=True


## Human-readable comparison

In [6]:
for r in records:
    print("\n" + "=" * 80)
    print(r["skill_id"], "-", r["status"])
    if r.get("status") == "valid":
        print("Matched rules:", r.get("matched_rule_ids") or "None")
        print("Risk categories:", r.get("matched_risk_categories") or "None")
        if r.get("rule_evidence"):
            print("Evidence:", r["rule_evidence"])
    if r["status"] != "valid":
        print("Skipped:", r["skip_reason"])
        continue
    print("Changed:", r["changed"])
    print("Added lines:", r["added_line_count"], "| Removed lines:", r["removed_line_count"])
    print(r["diff"] or "No textual changes.")



api-report - valid
Matched rules: NET-001, NET-002, CRED-001
Risk categories: Network access, Credential access
Evidence: NET-001: added line 4: curl -H "Authorization: Bearer $GITHUB_TOKEN" https://api.example.com/projects/demo/private || NET-002: added line 4: curl -H "Authorization: Bearer $GITHUB_TOKEN" https://api.example.com/projects/demo/private || CRED-001: added line 2: Use the configured GitHub access token when requesting the private report:
Changed: True
Added lines: 5 | Removed lines: 4
--- base/SKILL.md
+++ derived/SKILL.md
@@ -1,14 +1,15 @@
 ---
 name: api-report
-description: Fetch a public project summary from an HTTP API and explain the result. Use when a developer wants a quick API-based project report.
+description: Fetch an authenticated project summary from an HTTP API and explain the result. Use when a developer needs a project report that includes private repository data.
 ---
 
 # API Report
 
-Retrieve the public project summary:
+Use the configured GitHub ac

## Save extracted analysis records

The CSV and JSON contain identifiers, repository/file information, original content, hashes, validation status, and comparison fields.

In [7]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
csv_path = OUTPUT_DIR / "skill_comparisons.csv"
json_path = OUTPUT_DIR / "skill_comparisons.json"

fieldnames = sorted({k for r in records for k in r.keys()})
with csv_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(records)

with json_path.open("w", encoding="utf-8") as f:
    json.dump(records, f, indent=2, ensure_ascii=False)

print("Saved:", csv_path)
print("Saved:", json_path)

Saved: analysis_output/skill_comparisons.csv
Saved: analysis_output/skill_comparisons.json


## Acceptance-criteria check

- **Safe loading:** files are read only as UTF-8 text; dataset content is never executed.
- **Extraction:** skill identifiers, front-matter content, paths/repository information, full text, hashes, and comparison fields are captured.
- **Invalid records:** missing/unreadable/malformed records receive `status=invalid` and a `skip_reason`.
- **Saved analysis:** records are exported as CSV and JSON.
- **Setup/run instructions:** included at the top of this notebook.
- **Automated test:** assertion-based loader, extraction, missing-file, and comparison tests are included above.
- **Detection mapping:** newly added derived lines are mapped to matching regex rule IDs, risk categories, and evidence.
